In [1]:
library(tidyverse)
library(dbplyr)
library(bigrquery)
library(lubridate)

bq_auth()

project_id = "yhcr-prd-bradfor-bia-core"

# create connection to database
con <- DBI::dbConnect(bigrquery::bigquery(), 
                      project = project_id)

print(paste0("Connected to : ", project_id))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql




[1] "Connected to : yhcr-prd-bradfor-bia-core"


In [2]:
person_ids <- read.csv('data/additional_person_ids.csv', header = TRUE)

In [3]:
exclusion_data = "CB_2489.cb_ExclusionStudentsDetails"

exclusion_table <- tbl(con, exclusion_data) |>
    select(person_id,Year,NumberOfEnrolments,TotalFixedExclusions,TotalFixedSessions,PermanentExclusionCount,PrimarySENtype) 

In [4]:
exclusion_df <- collect(exclusion_table)

# Extract SEN type to compare with missing

In [5]:
exclusion_df |> distinct(PrimarySENtype) |> pull(PrimarySENtype)

[1] NA     " "    "BESD" "OTH"  "MLD"  "HI"   "SPLD" "SLCN" "PMLD" "ASD" 
[11] "PD"   "SLD"  "SEMH" "NSA"  "VI"   "MSI"

In [6]:
sen <- exclusion_df |>
    select(person_id, PrimarySENtype) |>
    filter(PrimarySENtype %in% c('BESD','OTH','MLD','HI','SPLD','SLCN','PMLD','ASD','PD','SLD','SEMH','NSA','VI','MSI'))


In [7]:
sen |> nrow()

[1] 10927

## Save csv

In [8]:
# save as csv
write.csv(sen, "data/additional_sen_extras.csv", row.names = FALSE)

# Exclusions

In [9]:
exclusion_df |> arrange(Year) |> distinct(Year) |> pull(Year)

[1] "2002" "2003" "2004" "2005" "2006" "2007" "2008" "2009" "2010" "2011"
[11] "2012" "2013" "2014" "2015" "2016" "2017" "2018" "2019" "2020" "2021"

In [10]:
exclusion_df |> arrange(NumberOfEnrolments) |> distinct(NumberOfEnrolments) |> pull(NumberOfEnrolments)

[1]  1  2  3  4 NA

In [11]:
exclusions_filtered <- exclusion_df |>
    filter(person_id %in% person_ids$person_id) |>
    select(-PrimarySENtype)

In [12]:
exclusions_filtered |> nrow()

[1] 4924

In [13]:
exclusions_filtered |> arrange(Year) |> distinct(Year) |> pull(Year)

[1] "2007" "2008" "2009" "2010" "2011" "2012" "2013" "2014" "2015" "2016"
[11] "2017" "2018" "2019"

In [14]:
exclusions_filtered_cohort <- exclusions_filtered |>
    left_join(person_ids, by = join_by(person_id))

Warning message in left_join(exclusions_filtered, person_ids, by = join_by(person_id)):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 4 of `x` matches multiple rows in `y`.
ℹ Row 8487 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”


In [15]:
head(exclusions_filtered_cohort)

person_id,Year,NumberOfEnrolments,TotalFixedExclusions,TotalFixedSessions,PermanentExclusionCount,NCCIS_ACADYR
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
F442350586B4F3CD0E0B5ADA701CE357664BDBD476679130440815D410CC0E61,2007,1,3,48,NA,2015/2016
FE7E0341560512618F68DB203ADB29FDF291C26185D50CB9A291C1CC4737FADD,2007,1,4,25,NA,2016/2017
DC4E7FC6B02A9C3E0D51398F4BB41B31ACD647847052DF06AC3710DE628DAE7B,2007,1,3,25,NA,2016/2017
1BFF9F036C8DD0C66E5330B88F6B5A2A7A7105809E2A94BFD6AF7B3A81F60399,2007,1,10,43,NA,2015/2016
1BFF9F036C8DD0C66E5330B88F6B5A2A7A7105809E2A94BFD6AF7B3A81F60399,2007,1,10,43,NA,2015/2016
2B61287808231C720C63753E882478D403B0E4F4A569D68AEFF37830BCCDEE60,2007,1,7,38,NA,2015/2016


## Select dates by cohort

In [16]:
cohort_1 <- exclusions_filtered_cohort |> filter(NCCIS_ACADYR == '2015/2016')

In [17]:
cohort_1 |> arrange(Year) |> distinct(Year) |> pull(Year)

[1] "2007" "2008" "2009" "2010" "2011" "2012" "2013" "2014" "2015" "2016"
[11] "2017" "2018"

In [18]:
cohort_1 <- cohort_1 |>
    filter(Year %in% c('2010','2011','2012','2013','2014','2015'))

In [19]:
cohort_2 <- exclusions_filtered_cohort |> filter(NCCIS_ACADYR == '2016/2017')

In [20]:
cohort_2 |> arrange(Year) |> distinct(Year) |> pull(Year)

[1] "2007" "2008" "2009" "2010" "2011" "2012" "2013" "2014" "2015" "2016"
[11] "2017" "2018" "2019"

In [21]:
cohort_2 <- cohort_2 |>
    filter(Year %in% c('2011','2012','2013','2014','2015','2016'))

In [22]:
cohort_2 |> arrange(person_id, Year)

person_id,Year,NumberOfEnrolments,TotalFixedExclusions,TotalFixedSessions,PermanentExclusionCount,NCCIS_ACADYR
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
0025514088C52FB45B7439689844A1FEA2FB8B2753994F432CC7DDC68036BD59,2016,1,1,10,0,2016/2017
004D0B5FF90E252B2912022950FEC4929A1ACB81BA61A626F3093B3B496C93CA,2014,1,1,6,0,2016/2017
00AAB0713CDD0D6705388B5DAED9E7AC7AA19E168DF615318DD6ED3DC4EE4CA1,2014,1,1,4,0,2016/2017
00C823B541B2D9EAA0C42DE865BD315331166CFCBCC6334744ADB31E5F72D916,2012,1,1,2,0,2016/2017
00D0DCE31E1E532A9AD051B9A03E286EE87258F21A7AC2C7EC65C9A2F2C3D6BB,2015,1,2,6,0,2016/2017
00D0DCE31E1E532A9AD051B9A03E286EE87258F21A7AC2C7EC65C9A2F2C3D6BB,2016,1,4,7,0,2016/2017
018B1258C2F1AEE4098F69728BCD61326D343AC119B826BE21430E8D7DAD2857,2015,1,1,24,0,2016/2017
01D5BB808C605F3206FF17738BC4011F459A59BBA3DD18AB15D8364CD1B383EF,2014,1,2,6,0,2016/2017
01D5BB808C605F3206FF17738BC4011F459A59BBA3DD18AB15D8364CD1B383EF,2014,1,2,6,0,2016/2017


## Rejoin cohorts and tally exclusions

In [23]:
all_exclusions <- rbind(cohort_1,cohort_2)

In [24]:
exclusions_alltime <- all_exclusions |> 
    group_by(person_id) %>%
      summarise(
        Suspensions = sum(TotalFixedExclusions, na.rm = TRUE),
        SuspensionSessions = sum(TotalFixedSessions, na.rm = TRUE),
        Exclusions = sum(PermanentExclusionCount, na.rm = TRUE),
        .groups = "drop"
      )

In [25]:
exclusions_alltime

person_id,Suspensions,SuspensionSessions,Exclusions
<chr>,<dbl>,<dbl>,<dbl>
00211975518FD5336E43850D59B5F5DDA7CD98B212760111CB40528BDECF3B1B,5,30,0
0025514088C52FB45B7439689844A1FEA2FB8B2753994F432CC7DDC68036BD59,1,10,0
0045ADA48CA299BC4445318E8D951610DCB89A829665CFC821E92340F0F8B30D,6,14,0
004D0B5FF90E252B2912022950FEC4929A1ACB81BA61A626F3093B3B496C93CA,1,6,0
007570ED5702694AEE71FBCA3ED93584AD7C19E06FE4CB64203ED8BE71B024EA,20,74,0
009354452C32CB442E4EEE6DF777F1EA4756148684FD6E3F75EA15F5F0E28081,5,38,0
0093549A2B3FB44847A7880014804DB333707601AE77A4AF949EE71E538194D6,7,13,0
00AAB0713CDD0D6705388B5DAED9E7AC7AA19E168DF615318DD6ED3DC4EE4CA1,1,4,0
00C823B541B2D9EAA0C42DE865BD315331166CFCBCC6334744ADB31E5F72D916,1,2,0


## Save csv

In [26]:
# save as csv
write.csv(exclusions_alltime, "data/additional_exclusions.csv", row.names = FALSE)